# 🕉️ ShlokaSage — QLoRA Fine-Tuning on Bhagavad Gita

Fine-tunes Llama 3.1 8B Instruct using QLoRA via Unsloth on Kaggle/Colab free GPU.

**Prerequisites:**
- Upload `train.jsonl` and `test.jsonl` to your Kaggle dataset or Colab files
- Kaggle: Use T4 x2 or P100 GPU accelerator
- Colab: Use T4 GPU runtime

In [ ]:
# ─── Step 1: Install Dependencies ───────────────────────────
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
# ─── Step 2: Login to HuggingFace (for saving model) ──────
from huggingface_hub import notebook_login
notebook_login()  # paste your HF token

In [ ]:
# ─── Step 3: Load Base Model with Unsloth ─────────────────
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096  # enough for our QA pairs
dtype = None  # auto-detect (float16 for T4)
load_in_4bit = True  # QLoRA 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# ─── Step 4: Add LoRA Adapters ────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,            # Unsloth optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ─── Step 5: Load Dataset ─────────────────────────────────
from datasets import load_dataset

# Option A: From local files (upload to Kaggle/Colab first)
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",   # adjust path as needed
    "test": "test.jsonl",
})

# Option B: From HuggingFace Hub (if you pushed with 03_format_dataset.py)
# dataset = load_dataset("your-username/ShlokaSage-dataset")

print(f"Train: {len(dataset['train'])} examples")
print(f"Test:  {len(dataset['test'])} examples")
print(f"\nSample:")
print(dataset['train'][0]['messages'])

In [ ]:
# ─── Step 6: Format Dataset for Training ──────────────────
# Apply chat template to convert messages into model's expected format

def apply_chat_template(example):
    """Convert messages list to formatted text using tokenizer's chat template."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_dataset = dataset.map(apply_chat_template)

# Check a sample
print(formatted_dataset['train'][0]['text'][:500])
print("...")

# Check token lengths
sample_tokens = tokenizer(formatted_dataset['train'][0]['text'], return_length=True)
print(f"\nSample token length: {sample_tokens['length'][0]}")

In [ ]:
# ─── Step 7: Configure Training ──────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # set True if short examples, saves time
    args=TrainingArguments(
        # Core
        output_dir="./shlokasage-checkpoints",
        num_train_epochs=3,
        
        # Batch size (adjust if OOM)
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 2*4 = 8
        
        # Learning rate
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        
        # Precision
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        
        # Logging & Eval
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        
        # Misc
        optim="adamw_8bit",
        seed=42,
        report_to="none",  # set to "wandb" if you want tracking
    ),
)

In [ ]:
# ─── Step 8: Check GPU Memory Before Training ────────────
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.max_memory_reserved() / 1024**3
max_memory = gpu_stats.total_mem / 1024**3
print(f"GPU: {gpu_stats.name}")
print(f"Memory: {reserved:.1f}GB reserved / {max_memory:.1f}GB total")
print(f"Available: {max_memory - reserved:.1f}GB")

In [ ]:
# ─── Step 9: Train! ──────────────────────────────────────
trainer_stats = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# ─── Step 10: Plot Training Loss ─────────────────────────
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [(x['step'], x['loss']) for x in log_history if 'loss' in x]
eval_losses = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label='Train Loss', alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label='Eval Loss', marker='o', markersize=4)

ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.set_title('ShlokaSage Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── Step 11: Quick Test ─────────────────────────────────
# Enable inference mode
FastLanguageModel.for_inference(model)

test_questions = [
    "What does Bhagavad Gita Chapter 2 Verse 47 mean?",
    "I am stressed about my exam results. What does the Gita say?",
    "Which verses talk about controlling anger?",
    "What was happening when Krishna showed his Vishwaroop?",
    "Explain the concept of Nishkama Karma from the Gita.",
]

system_prompt = """You are ShlokaSage, an expert guide on the Bhagavad Gita. You have deep knowledge of all 700 verses, their Sanskrit text, transliterations, and interpretations from major scholars including Shankaracharya, Ramanuja, Swami Sivananda, Swami Ramsukhdas, Swami Chinmayananda, and A.C. Bhaktivedanta Swami Prabhupada.

When answering questions:
- Always include the relevant Sanskrit shloka and transliteration when discussing a specific verse
- Cite specific chapter and verse numbers (e.g., BG 2.47)
- Weave in insights from scholars naturally, naming them when their perspective adds unique value
- Cross-reference related verses when helpful
- For life situations, map them to the most relevant verses with practical explanations
- Be warm and accessible, like a knowledgeable teacher — not an academic paper"""

for question in test_questions:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"A: {response}")

In [ ]:
# ─── Step 12: Save Model ─────────────────────────────────
HF_USERNAME = "your-username"  # ← CHANGE THIS
MODEL_NAME = "ShlokaSage-Llama3.1-8B-QLoRA"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

# Save LoRA adapter only (small, ~50-100MB)
model.save_pretrained(MODEL_NAME)
tokenizer.save_pretrained(MODEL_NAME)
print(f"Saved adapter locally to ./{MODEL_NAME}")

# Push adapter to HuggingFace Hub
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

In [ ]:
# ─── Step 13: (Optional) Merge & Save Full Model ─────────
# Merges LoRA adapter into base model for easier deployment
# This creates a larger file (~5-6GB in 4-bit GGUF)

# Save as 16-bit merged model
model.save_pretrained_merged(
    f"{MODEL_NAME}-merged",
    tokenizer,
    save_method="merged_16bit",
)

# Save as GGUF for llama.cpp / Ollama deployment
model.save_pretrained_gguf(
    f"{MODEL_NAME}-GGUF",
    tokenizer,
    quantization_method="q4_k_m",  # good balance of quality/size
)

# Push GGUF to Hub
model.push_to_hub_gguf(
    f"{REPO_ID}-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF model pushed to Hub!")